In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

# Membuat data sintetis: 300 titik, 4 kelompok alami.
X, _ = make_blobs(n_samples=300, centers=4, random_state=42)

# --- K-Means ---
# Membagi data ke 4 cluster berdasarkan jarak ke centroid terdekat.
kmeans = KMeans(n_clusters=4, random_state=42)
labels_k = kmeans.fit_predict(X)  # fit + langsung dapatkan label cluster
print("Inertia:", round(kmeans.inertia_, 2))          # total jarak ke centroid (makin kecil makin baik)
print("Silhouette:", round(silhouette_score(X, labels_k), 3))  # -1 s/d 1 (makin tinggi makin baik)

# --- Hierarchical ---
# Menggabungkan titik-titik terdekat secara bertahap dari bawah ke atas.
agg = AgglomerativeClustering(n_clusters=4)
labels_h = agg.fit_predict(X)
print("Silhouette Hierarchical:", round(silhouette_score(X, labels_h), 3))

Inertia: 564.91
Silhouette: 0.792
Silhouette Hierarchical: 0.792


In [5]:
# ============================================================
# TUGAS PRAKTIKUM PERTEMUAN 07
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_blobs

#Lakukan clustering pada dataset pelanggan (bebas atau buat sintetis).
# 1. MENYIAPKAN DATASET (SINTETIS PENGUIN)
# Meniru karakteristik 3 spesies: Adelie, Chinstrap, Gentoo
centers = [
    [38.8, 18.3, 190, 3700], # Karakteristik Adelie
    [48.8, 18.4, 195, 3733], # Karakteristik Chinstrap
    [47.5, 15.0, 217, 5076]  # Karakteristik Gentoo
]
X, _ = make_blobs(n_samples=344, centers=centers, cluster_std=2.0, random_state=42)

# Konversi ke DataFrame
df = pd.DataFrame(X, columns=['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g'])

# Scaling Data (Wajib untuk K-Means agar fitur berat badan tidak mendominasi)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

In [6]:
df.head()

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
0,37.149006,17.657228,190.825863,3698.872551
1,50.298711,16.849267,217.119261,5074.706126
2,42.531549,19.247666,187.617393,3701.313107
3,39.361984,17.054601,189.583755,3699.013998
4,38.294864,15.804434,193.264823,3697.139717


Tugas 1 : Lakukan clustering pada dataset pelanggan (bebas atau buat sintetis).

Tujuan tugas ini untuk menyiapkan dataset simulasi penguin saya, yang bakal kita kelompokkan secara otomatis. Saya ingin mempraktikkan cara kerja algoritma K-Means dalam mencari kemiripan antar data tanpa adanya label jawaban.

Langkah kerja :
1. Buat dataset buatan, menggunakan make_blobs untuk menciptakan dataset sitetis yang meniru karakteristik fisik tiga spesies penguin.
2. mengubah data mentah tadi menjadi format tabel pandas agar mudah diolah.
3. Memperhatikan bahwa fitur body_mass_g punya angak ribuan, sedangkan bill_depth_mm cuma belasan.
4. Menggunakan StandardScaler untuk samakan semua fitur kedalam skala yang setara(0 dan 1)
5. Menyimpan hasil scaling kedalam variabel X_scaled untuk lanjut ketahap clustering.

Penjelasan kode :
1. make_blobs() : ini fungsi buat bikin data acak tapi tetap terpusat di koordinat tertentu. saya setting pusatnya (centers) sesuai rata-rata fisik penguin asli biar simulasi ini seperti nyata.
2. StandardScaler() : ini adalah kunci dari K-Means. karena itu algoritma berbasis jarak (euclidean), kalau berat badan 3000g digabung sama panjang sirip 190mm tanpa scaling, modelnya akan anggap berat badan jauh lebih penting cuma gara-gara angkanya besar. scaler ini bikin semua fitur jadi adil.
3. fit_transform(df) : sekali jalan, model akan mempelajari pola sebaran data sekaligus mengubah nilainya menjadi skala standar.

Kesimpulan singkat : tahap persiapan ini sangat penting karna algortima K-Means sangat sensitif terhadap perbedaan skala. dengan melakukan standard scaling, saya memastikan bahwa setiap fitur fisik penguin punya pengaruh yang sama dalam menentukan kelompoknya nanti, sehingga hasil pengelompokan (clustering) yang dihasilkan lebih akurat.

In [8]:
# 2 : PROFIL RINGKAS TIAP CLUSTER

# Menjalankan K-Means
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

# Membuat profil ringkas (rata-rata tiap cluster)
profil_cluster = df.groupby('cluster').mean()

print("--- OUTPUT SOAL 2: PROFIL RINGKAS ---")
print(profil_cluster)

--- OUTPUT SOAL 2: PROFIL RINGKAS ---
         bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g
cluster                                                               
0             47.781842      14.932527         217.045877  5076.054056
1             38.794930      18.387915         189.964606  3700.000830
2             48.683093      18.454294         195.191232  3733.218336


Tugas 2 : Buat profil ringkas tiap cluster (misal: rata-rata pemasukan/pengeluaran).

Tujuan tugas ini untuk melakukan profiling. saya ingin melihat rata-rata nilai fisik dari setiap cluster agar saya bisa mengidentifikasi karakteristik unik tiap kelompok.

Langkah kerja
1. menjalankan algortima k-means dengan menentukan jumlah n_clisters=3 (sesuai dengan jumlah spesies penguin yang disimulasikan).
2. memasukkan hasil prediksi kelompok kedalam kolom baru bernama cluster di dataframe asli.
3. mengelompokkan data berdasarkan label clusternya, lalu menghitung nilai rata-rata untuk setiap fitur fisiknya.
4. menampilkan tabel profil ringkas tersebut.

Penjelasan kode, saya menggunakan logika analisis data yang sangat praktis :
1. n_clusters=3 : saya menetapkan angka 3 karena dari awal dataset ini berdasarkan 3 spesies penguin.
2. fit_predict(X_scaled) : fungsi ini melakukan dua hal sekaligus yaitu menyuruh model belajar dari data yang sudah di scale fit, lalu langsung memberikan label kelompok pada setiap baris data tersebut predict.
3. df.groupby('cluster').mean() : ini bagian untuk analisis. fungsi ini mengumpulkan semua anggota dicluster yang sama, lali merata-ratakan ukurannya. dengan begini dapat dilihat identitas setiap kelompok.

Kesimpulan singkat : dari tugas profiling ini, saya jadi paham kalau K-Means bukan cuma asal mengelompokkan data secara acak.  Hasil rata-ratanya menunjukkan perbedaan yang jelas antar kelompok; ada cluster yang mewakili penguin bertubuh besar dengan sayap panjang, dan ada cluster yang berisi penguin dengan paruh yang lebih pendek namun tebal. Ini membuktikan bahwa model saya berhasil menangkap perbedaan karakteristik fisik antar spesies meskipun tanpa bantuan label jawaban di awal.

In [9]:
# 3: MEMBERI NAMA SEGMEN

# Membuat mapping untuk penamaan berdasarkan profil rata-rata
mapping_segmen = {
    0: "Penguin Raksasa (Gentoo-like)",
    1: "Penguin Kecil (Adelie-like)",
    2: "Penguin Paruh Panjang (Chinstrap-like)"
}

# Mengaplikasikan nama ke dalam DataFrame
df['Nama_Segmen'] = df['cluster'].map(mapping_segmen)

print("--- OUTPUT SOAL 3: NAMA SEGMEN ---")
# Menampilkan 5 data unik untuk membuktikan segmen sudah terbentuk
print(df[['cluster', 'Nama_Segmen']].drop_duplicates().sort_values('cluster').to_string(index=False))

--- OUTPUT SOAL 3: NAMA SEGMEN ---
 cluster                            Nama_Segmen
       0          Penguin Raksasa (Gentoo-like)
       1            Penguin Kecil (Adelie-like)
       2 Penguin Paruh Panjang (Chinstrap-like)


Tugas 3 : Beri nama segmen (contoh: "Nilai tinggi", "Anggaran").

Tujuan tahap ini untuk terjemahkan angka kelompok (cluster 0,1 dan 2) yang dihasilkam oleh K-means menjadil label nama yang mudah dipahami manusia.

Langkah kerja :
1. berdasarkan mean dari tugas 2, saya identifikasi karakteristik fisik menonjol dari masing-masing cluster.
2. menyusun sebuah struktur data di python untuk memasangkan angka cluster dengan nama segmen yang sesuai (misalnya, cluster badan besar diberi nama Penguin Raksasa).
3. membuat kolom baru bernama Nama_Segmen di dalam DataFrame dan mengisi nilainya dengan hasil terjemahan dari kolom cluster menggunakan fungsi map
4. tampilkan sampel data unik untuk buktikan bahwa label segmen sudah berhasil terpasang pada masing-masing angka cluster dengan tepat.

Penjelasan kode, saya menggunakan teknik manipulasi data Pandas yang sangat efisien:
1. mapping_segmen = {...}: Ini adalah kamus (dictionary) manual yang saya buat. Angka di sebelah kiri adalah key (hasil prediksi K-Means), dan teks di sebelah kanan adalah value (nama alias yang saya berikan berdasarkan profil datanya).
2. df['cluster'].map(mapping_segmen): Ini adalah jurus andalan di Pandas. Fungsi map akan secara otomatis mengecek setiap baris di kolom cluster, melihat angkanya, lalu menggantinya dengan nama segmen dari kamus yang sudah saya buat, untuk kemudian disimpan di kolom baru Nama_Segmen.
3. drop_duplicates(): Karena dataset ini punya ratusan baris yang labelnya berulang, fungsi ini dipakai untuk membuang baris duplikat saat di-print, sehingga yang tampil di layar murni hanya contoh unik dari tiap cluster.
4. sort_values('cluster'): Mengurutkan tampilan data dari cluster 0, 1, lalu 2 agar rapi dan enak dibaca.
5. to_string(index=False): Trik kecil untuk menyembunyikan nomor indeks bawaan Pandas saat dicetak ke layar, biar tabelnya kelihatan lebih bersih.

Kesimpulan singkat : Pemberian nama segmen ini, saya berhasil menyempurnakan alur kerja Unsupervised Learning. Model K-Means yang pada awalnya bekerja secara (hanya mengelompokkan data berdasarkan jarak matematis menjadi angka 0, 1, 2) kini memiliki output akhir yang mudah dipahami. Teknik mapping ini sangat efektif untuk mengubah hasil komputasi mentah menjadi sebuah insight data yang siap dibaca oleh pengguna awam.

In [10]:
# 4 : 3 REKOMENDASI BISNIS/RISET

print("--- OUTPUT SOAL 4: REKOMENDASI BISNIS/RISET ---")

# Menyimpan rekomendasi dalam list dan mencetaknya ke output
rekomendasi = [
    "1. Segmen Penguin Raksasa: Gunakan alat pelacak (GPS) berukuran lebih besar dan tahan tekanan air dalam untuk riset migrasi.",
    "2. Segmen Penguin Kecil: Prioritaskan perlindungan area pantai dangkal karena ukuran tubuh yang kecil membuat kelompok ini lebih rentan.",
    "3. Segmen Penguin Paruh Panjang: Lakukan studi pola makan khusus karena struktur paruh yang panjang menunjukkan target mangsa yang berbeda."
]

for rek in rekomendasi:
    print(rek)

--- OUTPUT SOAL 4: REKOMENDASI BISNIS/RISET ---
1. Segmen Penguin Raksasa: Gunakan alat pelacak (GPS) berukuran lebih besar dan tahan tekanan air dalam untuk riset migrasi.
2. Segmen Penguin Kecil: Prioritaskan perlindungan area pantai dangkal karena ukuran tubuh yang kecil membuat kelompok ini lebih rentan.
3. Segmen Penguin Paruh Panjang: Lakukan studi pola makan khusus karena struktur paruh yang panjang menunjukkan target mangsa yang berbeda.


Tugas 4 : Tulis 3 rekomendasi bisnis berdasarkan cluster.

Tujuan merumuskan rekomendasi bisnis yang tepat sasaran untuk masing-masing segmen tersebut.

Langkah kerja
1. Mengingat kembali karakteristik tiap segmen dari tugas sebelumnya (Penguin Raksasa, Penguin Kecil, dan Penguin Paruh Panjang).
2. Menyusun 3 ide riset atau langkah perlindungan yang paling masuk akal dan relevan dengan fisik masing-masing tipe penguin.
3. Memasukkan kalimat-kalimat rekomendasi tersebut ke dalam struktur data List di Python agar tersimpan rapi.
4. Menggunakan perulangan (looping) untuk mengeksekusi dan menampilkan semua rekomendasi satu per satu ke layar output.

Penjelasan kode :
1. rekomendasi = [...]: Saya membuat sebuah List yang berisi tiga data bertipe string (teks). Masing-masing teks mewakili satu poin rekomendasi khusus yang sudah disesuaikan dengan hasil profiling segmen K-Means sebelumnya.
2. for rek in rekomendasi:: Ini adalah looping (perulangan) dasar. Daripada saya mengetik fungsi print() secara manual berulang-ulang, saya tinggal menyuruh Python untuk menelusuri isi List rekomendasi secara otomatis dari urutan pertama sampai terakhir.
3. print(rek): Mengeksekusi pencetakan teks ke layar pada setiap putaran looping.

Kesimpulan singkat : Tugas ini menjadi penutup yang sangat penting buat materi Clustering. Algoritma Unsupervised Learning seperti K-Means baru benar-benar berguna ketika hasil pengelompokan datanya bisa kita ubah menjadi strategi, keputusan bisnis, atau rekomendasi riset yang bisa diterapkan langsung di dunia nyata.